In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

df = pd.read_csv('marketing_clean.csv')

print(df.shape)
print(df.dtypes)
df[['AcceptedCmp3','AcceptedCmp4','AcceptedCmp5','AcceptedCmp1','AcceptedCmp2',
    'Response','AnyAccepted']].describe()

(2036, 32)
ID                       int64
Year_Birth               int64
Education                  str
Marital_Status             str
Income                 float64
Kidhome                  int64
Teenhome                 int64
Dt_Customer                str
Recency                  int64
MntWines                 int64
MntFruits                int64
MntMeatProducts          int64
MntFishProducts          int64
MntSweetProducts         int64
MntGoldProds             int64
NumDealsPurchases        int64
NumWebPurchases          int64
NumCatalogPurchases      int64
NumStorePurchases        int64
NumWebVisitsMonth        int64
AcceptedCmp3             int64
AcceptedCmp4             int64
AcceptedCmp5             int64
AcceptedCmp1             int64
AcceptedCmp2             int64
Complain                 int64
Response                 int64
Income_missing           int64
Age                      int64
AgeGroup                   str
Children                 int64
AnyAccepted              int

,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Response,AnyAccepted
count,2036.000000,2036.000000,2036.000000,2036.000000,2036.000000,2036.000000,2036.000000
mean,0.072692,0.076130,0.071709,0.065815,0.012770,0.153733,0.207760
std,0.259693,0.265271,0.258069,0.248020,0.112309,0.360781,0.405804
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


Response=1 비율이 15.4%, AnyAccepted=1 비율이 20.8%네요. 불균형이 심하진 않은거 확인

In [2]:
# H1: 캠페인을 한번이라도 참여했다면 Response의 오즈가 유의하게 높은가
model_h1 = smf.logit('Response ~ AnyAccepted', data=df).fit()
print(model_h1.summary())

# 오즈비로 변환 (계수 자체는 log-odds라 해석이 직관적이지 않음)
odds_ratio = np.exp(model_h1.params)
conf = np.exp(model_h1.conf_int())
conf['OR'] = odds_ratio
conf.columns = ['2.5%', '97.5%', 'OR']
print(conf)

Optimization terminated successfully.
         Current function value: 0.372337
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:               Response   No. Observations:                 2036
Model:                          Logit   Df Residuals:                     2034
Method:                           MLE   Df Model:                            1
Date:                Fri, 21 Aug 2026   Pseudo R-squ.:                  0.1323
Time:                        12:34:47   Log-Likelihood:                -758.08
converged:                       True   LL-Null:                       -873.71
Covariance Type:            nonrobust   LLR p-value:                 3.165e-52
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept      -2.3692      0.089    -26.614      0.000      -2.544      -2.195
AnyAccepted     2.0205    

오즈비 7.54 (95% CI: 5.81~9.79), p < 0.001 — AnyAccepted=1인 고객은 그렇지 않은 고객보다 6번째 캠페인 수락 오즈가 약 7.5배 높습니다. 단변량 모델치고 Pseudo R² 0.132도 꽤 큰 편이에요 (마케팅 반응 모델에서 R²가 0.1만 넘어도 강한 신호로 봅니다).

In [5]:
# 후보 변수들이 AnyAccepted와 관련 있는지 먼저 확인
from scipy import stats

candidates = ['Income', 'Age', 'Children', 'NumWebVisitsMonth', 
              'NumDealsPurchases', 'NumCatalogPurchases', 'Education']

for col in ['Income', 'Age', 'Children', 'NumWebVisitsMonth']:
    corr = df[col].corr(df['AnyAccepted'])
    print(f"{col} vs AnyAccepted: r = {corr:.3f}")

Income vs AnyAccepted: r = 0.309
Age vs AnyAccepted: r = 0.015
Children vs AnyAccepted: r = -0.225
NumWebVisitsMonth vs AnyAccepted: r = -0.120


In [6]:
for col in ['Income', 'Age', 'Children', 'NumWebVisitsMonth']:
    corr = df[col].corr(df['Response'])
    print(f"{col} vs Response: r = {corr:.3f}")

Income vs Response: r = 0.161
Age vs Response: r = -0.024
Children vs Response: r = -0.175
NumWebVisitsMonth vs Response: r = -0.002


변수	r (vs AnyAccepted)	r (vs Response)	교란변수 자격
Income	0.309	0.161	✅ 둘 다 있음 — 타당
Age	0.015	-0.024	❌ 둘 다 거의 0 — 애초에 넣을 근거 없었음
Children	-0.225	-0.175	✅ 둘 다 있음 — 타당
NumWebVisitsMonth	-0.120	-0.002	⚠️ 조건 절반만 충족 — 아래 설명

여기서 뭔가 이상한 게 보여야 정상이에요. Response와의 단순 상관은 r = -0.002로 사실상 0인데, H2 모델(다른 변수 통제한 상태)에서는 coef=0.2366, p<0.001로 강하게 유의했었죠. 이건 모순처럼 보이지만 실제로는 흔한 현상이에요 — 억제변수(suppressor variable) 효과라고 부릅니다.
즉, NumWebVisitsMonth는 "AnyAccepted를 설명하는 교란변수"라기보다는, "Response를 직접 예측하는 데 도움 되는 변수(다른 것들과 함께 봤을 때만)"에 가까워요. 교란변수 자격(조건 2)은 엄밀히는 통과 못 했지만, 그렇다고 모델에서 뺄 이유는 아니고 — 오히려 왜 이런 패턴이 나오는지 더 파볼 가치가 있는 신호예요.

Income, Children: 교란변수 조건 완전 충족, 계속 넣는 게 맞음
Age: 조건 미충족, 빼도 됨 (결과도 안 바뀔 가능성 높음)
NumWebVisitsMonth: 교란변수는 아니지만 억제효과로 보이는 흥미로운 변수, 유지하되 해석은 "교란 통제"가 아니라 "독립적 예측력"으로 프레이밍 바꿔야 함

H2의 목적을 다시 보면:

"AnyAccepted의 효과가 진짜인가, 아니면 원래 반응성 높은 고객 특성 때문인가?"

이 질문에 답하려면, 통제변수는 아무거나가 아니라 "AnyAccepted와도 관련 있고, Response와도 관련 있을 것 같은" 변수여야 해요. 이런 변수를 통계학에서 교란변수(confounder) 라고 부릅니다.

Income: 소득 높은 고객이 원래 구매력/관심도가 높아서 과거 캠페인에도, 이번 캠페인에도 둘 다 잘 반응할 수 있음
Age: 연령대에 따라 마케팅 채널 반응성이 다를 수 있음 (다만 결과는 유의하지 않았죠)
Children (Kidhome+Teenhome): 자녀 있는 가구는 지출 우선순위가 다르고, 이게 캠페인 전반의 반응성에 일관되게 영향 줄 수 있음
NumWebVisitsMonth: 온라인 채널 관여도가 높은 사람이 어느 캠페인이든 노출/반응 기회 자체가 많음 (구조적으로 반응 확률을 높이는 변수)

In [ ]:
# H2: 통제변수 추가 모델
# 참고: Age는 AnyAccepted(r=0.015), Response(r=-0.024) 둘 다와 상관이 거의 없어
# 교란변수 조건을 충족하지 못함. 실제로 모델에서도 유의하지 않게 나왔음(p=0.261).
# 결과에 영향 주는 변수는 아니지만, 참고용으로 모델에는 유지함.
model_h2 = smf.logit(model_h2 = smf.logit(
    'Response ~ AnyAccepted + Income + Age + Children + NumWebVisitsMonth',
    data=df
).fit()
print(model_h2.summary())

odds_ratio = np.exp(model_h2.params)
conf = np.exp(model_h2.conf_int())
conf['OR'] = odds_ratio
conf.columns = ['2.5%', '97.5%', 'OR']
print(conf)

Optimization terminated successfully.
         Current function value: 0.356400
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:               Response   No. Observations:                 2036
Model:                          Logit   Df Residuals:                     2030
Method:                           MLE   Df Model:                            5
Date:                Fri, 21 Aug 2026   Pseudo R-squ.:                  0.1695
Time:                        12:46:08   Log-Likelihood:                -725.63
converged:                       True   LL-Null:                       -873.71
Covariance Type:            nonrobust   LLR p-value:                 6.719e-62
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept            -3.8306      0.451     -8.491      0.000      -4.715      -2.946
AnyAccep

오즈비가 7.54 → 5.54로 떨어졌지만, 여전히 매우 크고 (p < 0.001) 유의합니다. Income, Age, Children, NumWebVisitsMonth를 다 통제해도 AnyAccepted의 효과가 완전히 사라지지 않고 27% 정도만 줄었어요.

→ H2는 지지됩니다.

나머지 변수들
Income (OR ≈ 1.00002): 유의(p<0.001)하지만 오즈비가 1에 거의 붙어있어 효과 크기 자체는 미미합니다. (소득 단위가 원 단위라 그런 거예요 — 1만원 단위로 스케일 바꾸면 더 읽기 좋아질 수 있어요.)
Age: p=0.261로 유의하지 않음 — 나이는 Response와 직접 연관이 없다는 뜻.
Children (OR ≈ 0.53): 유의하고, 자녀 있으면 오즈가 거의 절반으로 감소. 자녀 수가 늘수록 캠페인 반응이 낮아짐.
NumWebVisitsMonth (OR ≈ 1.27): 유의하고, 웹 방문 1회 늘 때마다 오즈 27% 증가.

In [ ]:
# VIF 체크 코드 (H2/H3에 쓴 변수 기준)
#  설명변수들끼리" 서로 얼마나 겹치는지 보는 지표
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
import pandas as pd

X = df[['AnyAccepted', 'Income', 'Age', 'Children', 'NumWebVisitsMonth']]
X = sm.add_constant(X)

vif_data = pd.DataFrame()
vif_data['Variable'] = X.columns
vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

print(vif_data)

            Variable        VIF
0              const  42.561933
1        AnyAccepted   1.151230
2             Income   1.944473
3                Age   1.077329
4           Children   1.289238
5  NumWebVisitsMonth   1.866904


const 행은 무시하고, 나머지 5개만 보시면 됩니다. 기준은 앞서 설명드린 대로: VIF 5 이하면 안심, 5~10이면 주의, 10 넘으면 문제 있는 변수로 봅니다.

전부 2 미만이에요. VIF 5 기준으로 보면 다중공선성 문제는 전혀 없다는 뜻입니다. 즉:

지난번 걱정했던 "Age가 유의하지 않은 게 Children이랑 겹쳐서 억눌린 거 아닌가?" → 아닙니다. VIF가 1.08로 사실상 독립적이니, Age의 p=0.261은 진짜로 "효과가 약하다"는 신호로 봐도 됩니다.
H2, H3의 계수들(Income, Children, NumWebVisitsMonth, AnyAccepted)도 서로 겹쳐서 왜곡된 게 아니라, 각자 독립적인 신호로 신뢰할 수 있어요.

정리: VIF 체크 결과 = H2 계수 해석에 대한 신뢰도 확인 완료. 이제 H3(교호작용) 결과 나오면 그것도 안심하고 해석하면 됩니다.

In [12]:
# 가설: AnyAccepted 효과는 특정 고객 특성과 결합해 더 강하게/약하게 나타난다
model_h3 = smf.logit(
    'Response ~ AnyAccepted * Children + Income + Age + NumWebVisitsMonth',
    data=df
).fit()
print(model_h3.summary())

odds_ratio = np.exp(model_h3.params)
conf = np.exp(model_h3.conf_int())
conf['OR'] = odds_ratio
conf.columns = ['2.5%', '97.5%', 'OR']
print(conf)

Optimization terminated successfully.
         Current function value: 0.355745
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:               Response   No. Observations:                 2036
Model:                          Logit   Df Residuals:                     2029
Method:                           MLE   Df Model:                            6
Date:                Fri, 21 Aug 2026   Pseudo R-squ.:                  0.1710
Time:                        14:27:04   Log-Likelihood:                -724.30
converged:                       True   LL-Null:                       -873.71
Covariance Type:            nonrobust   LLR p-value:                 1.462e-61
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept               -3.9799      0.463     -8.596      0.000      -4.887      -3.072

p = 0.105로, 0.05 기준에서는 유의하지 않습니다. 95% 신뢰구간(0.4705~1.0741)도 1을 포함하고 있어요 — "자녀 수에 따라 AnyAccepted 효과가 달라진다"고 통계적으로 확언할 수 없는 상태입니다.

→ H3는 (엄밀한 기준으로는) 기각. "AnyAccepted 효과는 자녀 수와 상관없이 균일하다"는 쪽에 더 가까워요

H3 : "이 관련성(AnyAccepted → Response)은 모든 고객에게 균일한가, 아니면 특정 고객 특성과 결합해 더 강하게 나타나는가?"

Children 단독 효과: p<0.001로 강하게 유의 (자녀 자체는 확실히 Response를 낮춤)
AnyAccepted:Children 교호작용: p=0.105로 유의하지 않음 (자녀가 AnyAccepted 효과의 "크기"까지 바꾸는지는 불확실)

즉 자녀가 1명 늘어날 때마다, Response 오즈가 원래의 53.3% 수준으로 줄어든다는 뜻이고, 이를 "감소분"으로 표현하면 약 47% 감소가 됩니다. (참고: 이 47% 감소는 Children의 단독 효과이고, 방금 얘기한 
AnyAccepted:Children 교호작용과는 별개의 수치입니다.)

In [ ]:
# H4 — 개별 캠페인 계수 비교 (최근성 효과)
# 직전 1개 캠페인(AcceptedCmp5)의 수락 여부가 그 이전 캠페인들보다 Response에 더 강한 개별 계수를 갖는다
model_h4 = smf.logit(
    'Response ~ AcceptedCmp1 + AcceptedCmp2 + AcceptedCmp3 + AcceptedCmp4 + AcceptedCmp5',
    data=df
).fit()
print(model_h4.summary())

odds_ratio = np.exp(model_h4.params)
conf = np.exp(model_h4.conf_int())
conf['OR'] = odds_ratio
conf.columns = ['2.5%', '97.5%', 'OR']
print(conf)

Optimization terminated successfully.
         Current function value: 0.356257
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:               Response   No. Observations:                 2036
Model:                          Logit   Df Residuals:                     2030
Method:                           MLE   Df Model:                            5
Date:                Fri, 21 Aug 2026   Pseudo R-squ.:                  0.1698
Time:                        14:29:24   Log-Likelihood:                -725.34
converged:                       True   LL-Null:                       -873.71
Covariance Type:            nonrobust   LLR p-value:                 5.039e-62
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept       -2.3322      0.084    -27.622      0.000      -2.498      -2.167
AcceptedCmp1     1.3139

In [10]:
result_table = conf.drop('Intercept').sort_values('OR', ascending=False)
print(result_table)

                  2.5%      97.5%        OR
AcceptedCmp3  4.448995   9.562907  6.522677
AcceptedCmp5  3.198789   7.531015  4.908169
AcceptedCmp2  1.371380  11.832674  4.028286
AcceptedCmp1  2.381581   5.812469  3.720600
AcceptedCmp4  1.157370   2.893397  1.829954


기각입니다. 가장 최근 캠페인(AcceptedCmp5)이 가장 강한 계수를 가질 거란 예상과 달리, AcceptedCmp3의 오즈비(6.52)가 AcceptedCmp5(4.91)보다 더 큽니다. "직전 캠페인일수록 효과가 크다"는 최근성 효과 패턴은 이 데이터에서 확인되지 않았어요.

최종 결론 한 줄 요약
과거 캠페인 참여 경험은 6차 캠페인 수락과 강하게 연관됨 (H1)
소득/자녀/웹방문 등을 통제해도 이 연관성은 상당 부분 유지됨 (H2)
다만 자녀 유무에 따라 이 효과 크기가 통계적으로 다르다는 근거는 부족함 (H3)
캠페인 3, 5차가 특히 강한 신호였고, "최근 캠페인일수록 효과가 크다"는 최근성 가설은 지지되지 않음 (H4)

In [16]:
# 연속된 캠페인 쌍: (t-1 수락 여부) → (t 수락 여부)
pairs = [
    ('AcceptedCmp1', 'AcceptedCmp2'),
    ('AcceptedCmp2', 'AcceptedCmp3'),
    ('AcceptedCmp3', 'AcceptedCmp4'),
    ('AcceptedCmp4', 'AcceptedCmp5'),
    ('AcceptedCmp5', 'Response'),
]

results = []
for prev, curr in pairs:
    model = smf.logit(f'{curr} ~ {prev}', data=df).fit(disp=0)
    or_val = np.exp(model.params[prev])
    p_val = model.pvalues[prev]
    results.append({'lag': f'{prev} → {curr}', 'OR': or_val, 'p_value': p_val})

lag_df = pd.DataFrame(results)
print(lag_df)

                           lag            OR       p_value
0  AcceptedCmp1 → AcceptedCmp2  1.326464e+01  1.621993e-10
1  AcceptedCmp2 → AcceptedCmp3  3.946479e+00  3.743078e-03
2  AcceptedCmp3 → AcceptedCmp4  2.612348e-09  9.970677e-01
3  AcceptedCmp4 → AcceptedCmp5  9.989880e+00  1.172182e-30
4      AcceptedCmp5 → Response  9.827018e+00  2.763636e-36


c:\anaconda\envs\myenv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [15]:
pd.crosstab(df['AcceptedCmp3'], df['AcceptedCmp4'])

AcceptedCmp4,0,1
AcceptedCmp3,,
0,1733,155
1,148,0


AcceptedCmp3=1이면서 AcceptedCmp4=1인 사람이 단 한 명도 없어요 (셀 값이 0). 이게 바로 "완전분리(perfect separation)"입니다 — AcceptedCmp3=1인 사람은 전부 AcceptedCmp4=0이었다는 뜻이에요.

이건 노이즈나 실수가 아니라, 실제 데이터 패턴이 이렇다는 뜻이에요. 즉 "3차 캠페인을 수락한 고객은 (148명 전원이) 4차 캠페인은 아무도 수락하지 않았다." 이건 우연이라기엔 너무 완벽해서, 마케팅 맥락에서 몇 가지 가설을 세워볼 수 있어요:

3차와 4차 캠페인이 거의 동시에 진행돼서 같은 사람에게 중복 오퍼가 안 갔을 가능성 (운영상 배제 규칙)
3차 캠페인 수락자는 쿨다운(cool-down) 기간으로 4차 캠페인 대상에서 아예 제외됐을 가능성
3차와 4차가 동일하거나 유사한 상품/오퍼라서 이미 받은 사람에게 재발송을 안 했을 가능성

Lag	결과
Cmp1 → Cmp2	OR 13.26, 유의
Cmp2 → Cmp3	OR 3.95, 유의
Cmp3 → Cmp4	추정 불가 (완전분리) — 3차 수락자는 4차 100% 미수락
Cmp4 → Cmp5	OR 9.99, 유의
Cmp5 → Response	OR 9.83, 유의

종합 해석: 전반적으로 "직전 캠페인 참여가 다음 캠페인 참여를 강하게 예측한다"는 lag 패턴이 확인되지만(4개 구간 중 3개), 3→4 구간은 이 패턴에서 완전히 벗어난 예외이자 그 자체로 별도로 짚을 만한 발견입니다.